# 05 — Model Training and Evaluation

Compare Logistic Regression (interpretable baseline) and Random Forest (non-linear candidate). Model selection uses validation PR-AUC. A final model is refit on train + validation and evaluated once on a later held-out test period.

A 90-day embargo is used between temporal partitions to reduce overlap between forward-looking labels and later periods. Metrics are only available after running the notebook on the actual dataset.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "train.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import FEATURES_FILE
from src.train import chronological_split, main as train_model, REPORT_DIR

if not FEATURES_FILE.exists():
    raise FileNotFoundError("Run Notebook 04 first to create customer_snapshots.parquet.")
data = pd.read_parquet(FEATURES_FILE)
data["snapshot_date"] = pd.to_datetime(data["snapshot_date"])
print("Rows:", len(data), "| Distinct snapshot months:", data.snapshot_date.nunique())

## 1. Inspect chronological partitions

In [ ]:
train, validation, test = chronological_split(data)
for name, part in [("train", train), ("validation", validation), ("test", test)]:
    print(name, len(part), part.snapshot_date.min(), "to", part.snapshot_date.max(),
          "| positive rate:", round(part.repeat_purchase_90d.mean(), 4))

## 2. Train, select and evaluate

The training module fits the two candidate pipelines, selects by validation PR-AUC, refits the selected model on development data, and evaluates on the untouched test period.

In [ ]:
train_model()
metrics_path = REPORT_DIR / "model_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Selected model:", metrics["selected_model"])
display(pd.DataFrame(metrics["test_metrics"], index=["held_out_test"]).T)
display(pd.DataFrame(metrics["capacity_metrics"]).T)

## 3. Interpretation

- PR-AUC is especially useful when positive and negative classes are imbalanced.
- Precision@10% and lift@10% estimate how concentrated future repeat purchasers are among the highest-scored customers.
- A 0.50 threshold is not automatically optimal for a business outreach programme.
- Compare results with a simple baseline and inspect performance across time before making operational claims.

Do not describe the output as a churn probability or as evidence that a campaign will cause a purchase.